# Imports

In [1]:

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import when
import config.ConnectionConfig as cc
from pyspark.sql.functions import expr, lag, monotonically_increasing_id, row_number
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# SetUp

In [2]:
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_USER", 4)
spark.getActiveSession()

25/04/23 11:54:58 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 10.140.98.193 instead (on interface wlp2s0)
25/04/23 11:54:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ab3aa3b7-2cc2-418b-ae08-ad22097e3a93;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# Extract

In [3]:
cc.set_connectionProfile("default")
df_users = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()

df_subscriptions = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()
# df_subscriptions_types = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", "subscription_types") \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "subscriptiontypeid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0) \
#     .option("upperBound", 10000) \
#     .load()

# Transform

In [4]:
#Ok so this joins the two subscriptions tables and then it creates that cute little expression, where for each day, month or year, it uses the date_add function with a start from validfrom. In theory, this should properly calculate the end date, give or take a few days.
df_subscriptions = df_subscriptions.withColumn(
    "end_date",
    expr("""
        CASE
            WHEN subscriptiontypeid = 1 THEN date_add(validfrom, 1)
            WHEN subscriptiontypeid = 2 THEN date_add(validfrom, 30)
            WHEN subscriptiontypeid = 3 THEN date_add(validfrom, 365)
        END
    """)
)

df_joined = df_users.join(df_subscriptions, "userid", "left")
df_joined.show()

+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+--------------+----------+------------------+----------+
|userid|                name|               email|              street|  number|zipcode|                city|country_code|subscriptionid| validfrom|subscriptiontypeid|  end_date|
+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+--------------+----------+------------------+----------+
|    12|        Simons Thijs|Thijs.Simons@outl...|         Bergenhoeve| 81 0302|   2040|Antwerpen/Berendr...|          BE|            23|2023-10-20|                 3|2024-10-19|
|    12|        Simons Thijs|Thijs.Simons@outl...|         Bergenhoeve| 81 0302|   2040|Antwerpen/Berendr...|          BE|            22|2021-09-09|                 3|2022-09-09|
|    12|        Simons Thijs|Thijs.Simons@outl...|         Bergenhoeve| 81 0302|   2040|Antwerpen/Berendr

In [10]:
#TRANSFORM
#SCD stuff
#Ok so, im writing this one down so I don't fucking forget
#This bitch organizes the data by user and the date when their address became valid(i.e. validfrom). The for each person it checks what their next address is going to be using lead() and then compare the current one to the "next" one. If any part of it changes, it's marked as a new address. Additionally the end date of the current address is set to one date before the new one starts.
#This only works if there's multiple user instances, which, upon further consideration, is not the case
# window_spec = Window.partitionBy("userid").orderBy("validfrom")
# df_dim_user = df_joined.withColumn("prev_street", lead("street").over(window_spec))\
#     .withColumn("prev_number", lead("number").over(window_spec))\
#     .withColumn("prev_zipcode", lead("zipcode").over(window_spec))\
#     .withColumn("prev_city", lead("city").over(window_spec))\
#     .withColumn("prev_country_code", lead("country_code").over(window_spec))\
#     .withColumn("next_start_date", lead("validfrom").over(window_spec))\
#     .withColumn("is_new_address", expr("""
#         (street <> prev_street) OR
#         (number <> prev_number) OR
#         (zipcode <> prev_zipcode) OR
#         (city <> prev_city) OR
#         (country_code <> prev_country_code)
#     """))\
#     .withColumn("final_end_date", expr("""
#         CASE WHEN is_new_address THEN date_sub(next_start_date, 1)
#         ELSE '9999-12-31' END
#     """))\
#     .drop(
#         "prev_street", "prev_number", "prev_zipcode", "prev_city",
#         "prev_country_code", "next_start_date", "is_new_address"
#     )

RuntimeError: SparkContext or SparkSession should be created first.

In [5]:


#This bitch works as the previous one, but it actually compares to any previous row/address
#idk anymore, kms
window_spec = Window.partitionBy("userid").orderBy("validfrom")
df_dim_user = df_joined \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .withColumn("current", F.when(F.col("row_num") == 1, True).otherwise(False)) \
    .drop("row_num") \
    .withColumn("prev_street", F.lag("street").over(window_spec))\
    .withColumn("prev_number", F.lag("number").over(window_spec))\
    .withColumn("prev_zipcode", F.lag("zipcode").over(window_spec))\
    .withColumn("prev_city", F.lag("city").over(window_spec))\
    .withColumn("prev_country_code", F.lag("country_code").over(window_spec))\
    .withColumn("is_new_address", F.expr("""
        (street <> prev_street) OR
        (number <> prev_number) OR
        (zipcode <> prev_zipcode) OR
        (city <> prev_city) OR
        (country_code <> prev_country_code)
    """)) \
    .drop(
        "prev_street", "prev_number", "prev_zipcode", "prev_city",
        "prev_country_code", "prev_validfrom", "is_new_address"
    )
df_dim_user.show()

+------+-----------------+--------------------+--------------------+-------+-------+--------------------+------------+--------------+----------+------------------+----------+-------+
|userid|             name|               email|              street| number|zipcode|                city|country_code|subscriptionid| validfrom|subscriptiontypeid|  end_date|current|
+------+-----------------+--------------------+--------------------+-------+-------+--------------------+------------+--------------+----------+------------------+----------+-------+
|    12|     Simons Thijs|Thijs.Simons@outl...|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|            21|2020-08-07|                 3|2021-08-07|  false|
|    12|     Simons Thijs|Thijs.Simons@outl...|         Bergenhoeve|81 0302|   2040|Antwerpen/Berendr...|          BE|            22|2021-09-09|                 3|2022-09-09|   true|
|    12|     Simons Thijs|Thijs.Simons@outl...|         Bergenhoeve|81 0302|   2040|A

# LOAD

In [6]:
df_dim_user = df_dim_user.select(
    monotonically_increasing_id().alias("user_sk"),
    df_dim_user.userid.alias("user_id"),
    df_dim_user.street,
    df_dim_user.number,
    df_dim_user.zipcode,
    df_dim_user.city,
    df_dim_user.country_code,
    df_dim_user.validfrom.alias("start_date"),
    df_dim_user.end_date.alias("end_date"),
    df_dim_user.subscriptionid,
    df_dim_user.validfrom,
    df_dim_user.subscriptiontypeid,
    df_dim_user.end_date.alias("end_subscription_date"),
    df_dim_user.current
)
df_dim_user.show()

+-------+-------+--------------------+--------+-------+--------------------+------------+----------+----------+--------------+----------+------------------+---------------------+-------+
|user_sk|user_id|              street|  number|zipcode|                city|country_code|start_date|  end_date|subscriptionid| validfrom|subscriptiontypeid|end_subscription_date|current|
+-------+-------+--------------------+--------+-------+--------------------+------------+----------+----------+--------------+----------+------------------+---------------------+-------+
|      0|      6|   Jan Ockegemstraat|168 0107|   2650|              Edegem|          BE|2019-06-29|2020-06-28|             8|2019-06-29|                 3|           2020-06-28|   true|
|      1|      6|   Jan Ockegemstraat|168 0107|   2650|              Edegem|          BE|2023-11-30|2024-11-29|             9|2023-11-30|                 3|           2024-11-29|  false|
|      2|      6|   Jan Ockegemstraat|168 0107|   2650|          

In [7]:
df_dim_user.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("dim_user")

25/04/23 11:36:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
spark.stop()